In [1]:

import torch
import torchvision
import numpy as np
import random

SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("GPU available:", torch.cuda.is_available())

print("Setup completed successfully!")


PyTorch version: 2.14.1+cpu
Torchvision version: 0.29.1+cpu
GPU available: False
Setup completed successfully!


In [2]:

import torch
from torchvision import datasets, transforms

transform = transforms.ToTensor()

train_data = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

print("Training samples:", len(train_data))
print("Testing samples:", len(test_data))
print("Image shape:", train_data[0][0].shape)
print("Number of classes:", len(train_data.classes))
print("Classes:", train_data.classes)


100%|██████████| 26.4M/26.4M [00:01<00:00, 13.4MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 203kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.78MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 9.24MB/s]

Training samples: 60000
Testing samples: 10000
Image shape: torch.Size([1, 28, 28])
Number of classes: 10
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


In [3]:

from torch.utils.data import random_split, DataLoader

train_size = 54000
val_size = 6000

train_set, val_set = random_split(
    train_data,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

print("Training samples:", len(train_set))
print("Validation samples:", len(val_set))
print("Testing samples:", len(test_data))


Training samples: 54000
Validation samples: 6000
Testing samples: 10000


In [4]:

import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 10)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader):.4f}")

print("Training completed!")


Epoch [1/5], Loss: 0.9882
Epoch [2/5], Loss: 0.6815
Epoch [3/5], Loss: 0.6143
Epoch [4/5], Loss: 0.5776
Epoch [5/5], Loss: 0.5538
Training completed!


In [5]:

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("Validation Accuracy:", round(accuracy, 2), "%")


Validation Accuracy: 81.43 %
